In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, max as spark_max
from delta.tables import DeltaTable

In [0]:
# Get the latest valid model year (excluding placeholder 9999)
max_year = (
    spark.table("bronze.model_years")
         .filter("model_year <> 9999")
         .select(spark_max("model_year").alias("y"))
         .collect()[0]["y"]
)


In [0]:
# Get distinct makes for the latest model year
makes = [r.make for r in (
    spark.table("bronze.vehicle_makes")
         .filter(f"model_year = {max_year}")
         .select("make")
         .distinct()
         .collect()
)]

In [0]:
# Fetch vehicle models for each make for the given model year from the NHTSA API
base_url = "https://api.nhtsa.gov/products/vehicle/models"
issue_type = "c"

rows = []
for make in makes:
    params = {"modelYear": int(max_year), "make": make, "issueType": issue_type}
    resp = requests.get(base_url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    for m in data.get("results", []):
        model = m.get("model")
        if model:
            rows.append(Row(
                model_year=int(max_year),
                make=make,
                model=model
            ))

# Create DataFrame of models, remove duplicates, and add ingestion metadata
df_models = spark.createDataFrame(rows) \
    .dropDuplicates(["model_year", "make", "model"]) \
    .withColumn("ingestion_ts", current_timestamp()) \
    .withColumn("source_url", lit(base_url))

In [0]:
# 3) MERGE into the bronze.vehicle_models table
target_table = "bronze.vehicle_models"

if spark.catalog.tableExists(target_table):
    # If the table exists, perform an upsert (merge) to update existing rows and insert new ones
    dt = DeltaTable.forName(spark, target_table)
    (dt.alias("t")
      .merge(df_models.alias("s"),
             "t.model_year = s.model_year AND t.make = s.make AND t.model = s.model")
      .whenMatchedUpdate(set={
          "ingestion_ts": "current_timestamp()",
          "source_url": "s.source_url"
      })
      .whenNotMatchedInsert(values={
          "model_year": "s.model_year",
          "make": "s.make",
          "model": "s.model",
          "ingestion_ts": "current_timestamp()",
          "source_url": "s.source_url"
      })
      .execute()
    )
else:
    # If the table does not exist, create it with the current data
    df_models.write.format("delta").mode("overwrite").saveAsTable(target_table)